# Notebook 03 — Visualisation des résultats de détection

Ce notebook exécute le pipeline complet sur la fixture synthétique et visualise
les résultats : mapping MITRE ATT&CK, z-scores par utilisateur, timeline des anomalies.

In [ ]:
import sys
sys.path.insert(0, '../src')

import csv
from datetime import timedelta
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from ueba.adapters.wazuh import WazuhAdapter
from ueba.domain.features import UEBAFeatureExtractor, FEATURE_NAMES
from ueba.domain.mitre import MitreMapper

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Exécution du pipeline

In [ ]:
FIXTURE = Path('../tests/integration/fixtures/sample_logs.csv')

with FIXTURE.open(newline='', encoding='utf-8') as f:
    records = list(csv.DictReader(f))

events = WazuhAdapter().normalize(records)
extractor = UEBAFeatureExtractor(
    window_size=timedelta(hours=1),
    window_step=timedelta(minutes=30),
)
vectors = extractor.extract(events)

# Fenêtres de spray (16 mai 14h, failed_login > 0)
spray_vectors = [
    v for v in vectors
    if v.window_start.day == 16 and v.window_start.hour == 14
    and v.failed_login_count > 0
]

mapper = MitreMapper()
mitre_matches = mapper.match_population(spray_vectors)

print(f'Vecteurs totaux : {len(vectors)}')
print(f'Vecteurs spray : {len(spray_vectors)}')
print(f'\nMatches MITRE ATT&CK :')
for m in mitre_matches:
    print(f'  [{m.technique_id}] {m.technique_name} — {m.rationale}')

## 2. Carte thermique des features pour les fenêtres de spray

In [ ]:
spray_rows = []
for v in spray_vectors:
    row = {'user': v.user}
    for name in FEATURE_NAMES:
        row[name] = getattr(v, name)
    spray_rows.append(row)

spray_df = pd.DataFrame(spray_rows).set_index('user')

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    spray_df[list(FEATURE_NAMES)].T, annot=True, fmt='.1f',
    cmap='YlOrRd', linewidths=0.4, ax=ax
)
ax.set_title('Features comportementales — fenêtres de Password Spray (16 mai 14h)')
ax.set_xlabel('Utilisateur')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()

## 3. Résumé : failed_login_count par utilisateur (16 mai)

In [ ]:
rows = []
for v in vectors:
    rows.append({
        'user': v.user,
        'window_start': v.window_start,
        'day': v.window_start.day,
        'failed_login_count': v.failed_login_count,
    })
all_df = pd.DataFrame(rows)

day16 = all_df[all_df['day'] == 16].copy()
summary = day16.groupby('user')['failed_login_count'].max().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['firebrick' if v >= 2 else 'steelblue' for v in summary.values]
summary.plot(kind='bar', ax=ax, color=colors)
ax.axhline(2, color='red', linestyle='--', alpha=0.7, label='Seuil spray (≥2)')
ax.set_title('Max failed_login_count par utilisateur — 16 mai 2026')
ax.set_xlabel('Utilisateur')
ax.set_ylabel('failed_login_count (max sur fenêtre)')
ax.legend()
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

spray_users = summary[summary >= 2].index.tolist()
print(f'Utilisateurs avec ≥2 échecs (seuil spray) : {spray_users}')

## 4. Affichage du match MITRE T1110.003

In [ ]:
spray_match = next((m for m in mitre_matches if m.technique_id == 'T1110.003'), None)

if spray_match:
    print('=' * 70)
    print(f'TECHNIQUE    : {spray_match.technique_id} — {spray_match.technique_name}')
    print(f'TACTIQUE     : {spray_match.tactic}')
    print(f'SOURCE       : {spray_match.source}')
    print(f'RATIONALE    : {spray_match.rationale}')
    print('=' * 70)
else:
    print('T1110.003 non détecté — vérifier le seuil PASSWORD_SPRAY_MIN_USERS')